# FitNutrition: Personalized Nutrition Tracking and Dietary Analytics System
**Course Project — Python**

## Abstract
FitNutrition is an educational prototype for logging meals, calculating nutritional intake, comparing foods, and analyzing dietary patterns. It demonstrates Python data analytics, SQLite database integration, and data visualization. The companion Streamlit application provides an interactive interface.

> **Data note:** The included food values are approximate illustrative values for demonstration, not a substitute for verified product labels or professional nutrition advice. Values depend on brand, preparation, and serving size.


## 1. Problem Statement and Objectives
Manually tracking food intake can make it difficult to understand daily macronutrient intake and longer-term dietary patterns. FitNutrition provides a small food database, meal logging, nutrient calculations, food comparisons, and weekly summaries.

**Objectives**
1. Store nutritional information in a structured dataset.
2. Calculate calories, protein, carbohydrates, and fats for recorded food quantities.
3. Persist meal records using SQLite.
4. Compare food items on a consistent serving basis.
5. summarize daily and weekly intake with visualizations.
6. Demonstrate an interactive user interface through Streamlit.


## 2. Libraries and Dataset
The notebook uses Pandas for tabular processing, SQLite for persistence, and Matplotlib for plots. The companion app uses Streamlit and Plotly. `food_data.csv` contains a small sample dataset with nutrient values per 100 g and typical serving sizes.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("food_data.csv")
if not DATA_PATH.exists():
    # Allows the notebook to be run from the project folder.
    raise FileNotFoundError("Place food_data.csv in the same folder as this notebook.")

foods = pd.read_csv(DATA_PATH)
foods.head()


## 3. Data Validation
Check column types, missing values, and duplicate food names before using the dataset.

In [ ]:
required_columns = [
    "food_id", "food_name", "category", "serving_g",
    "calories_kcal_per_100g", "protein_g_per_100g",
    "carbs_g_per_100g", "fat_g_per_100g"
]
missing_columns = sorted(set(required_columns) - set(foods.columns))
assert not missing_columns, f"Missing columns: {missing_columns}"

print("Dataset rows:", len(foods))
print("Missing values by column:\n", foods.isna().sum())
print("Duplicate food names:", foods["food_name"].duplicated().sum())
assert foods["food_name"].notna().all()
assert foods["food_name"].is_unique
assert (foods[["serving_g", "calories_kcal_per_100g", "protein_g_per_100g",
               "carbs_g_per_100g", "fat_g_per_100g"]] >= 0).all().all()


## 4. Nutrient Calculation
Nutrient values are stored per 100 g. For a quantity `q` grams, the estimated nutrient value is `value_per_100g × q / 100`.

In [ ]:
def calculate_nutrients(food_row, quantity_g):
    if quantity_g <= 0:
        raise ValueError("Quantity must be greater than zero.")
    factor = quantity_g / 100.0
    return {
        "calories_kcal": round(float(food_row["calories_kcal_per_100g"]) * factor, 2),
        "protein_g": round(float(food_row["protein_g_per_100g"]) * factor, 2),
        "carbs_g": round(float(food_row["carbs_g_per_100g"]) * factor, 2),
        "fat_g": round(float(food_row["fat_g_per_100g"]) * factor, 2),
    }

sample_food = foods.loc[foods["food_name"] == "Boiled egg"].iloc[0]
calculate_nutrients(sample_food, 100)


## 5. Food Comparison Analytics
Compare selected foods per 100 g. This consistent basis is useful for exploratory comparison, although nutritional quality cannot be reduced to one number.

In [ ]:
comparison = foods[[
    "food_name", "calories_kcal_per_100g", "protein_g_per_100g",
    "carbs_g_per_100g", "fat_g_per_100g"
]].sort_values("protein_g_per_100g", ascending=False)

comparison.head(10)


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(comparison["food_name"], comparison["protein_g_per_100g"])
plt.title("Protein content by food (per 100 g)")
plt.ylabel("Protein (g)")
plt.xticks(rotation=70, ha="right")
plt.tight_layout()
plt.show()


## 6. SQLite Database Integration
The following example creates tables and saves a sample meal entry. The Streamlit app uses the same general schema and adds meal logging, target management, food addition, and history views.

In [ ]:
DB_PATH = Path("nutrition_demo.db")
with sqlite3.connect(DB_PATH) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS foods (
            food_id INTEGER PRIMARY KEY,
            food_name TEXT UNIQUE,
            category TEXT,
            serving_g REAL,
            calories_kcal_per_100g REAL,
            protein_g_per_100g REAL,
            carbs_g_per_100g REAL,
            fat_g_per_100g REAL
        )
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS meals (
            meal_id INTEGER PRIMARY KEY AUTOINCREMENT,
            meal_date TEXT, meal_type TEXT, food_name TEXT, quantity_g REAL,
            calories REAL, protein_g REAL, carbs_g REAL, fat_g REAL
        )
    """)
    count = con.execute("SELECT COUNT(*) FROM foods").fetchone()[0]
    if count == 0:
        foods.to_sql("foods", con, if_exists="append", index=False)

with sqlite3.connect(DB_PATH) as con:
    saved_foods = pd.read_sql_query("SELECT food_name, category FROM foods LIMIT 5", con)
saved_foods


## 7. Example Meal Log and Daily Summary
This cell creates a reproducible in-memory example rather than changing any real user's data.

In [ ]:
example_entries = [
    ("Breakfast", "Oats, dry", 40),
    ("Breakfast", "Milk, whole", 200),
    ("Snack", "Banana", 120),
    ("Lunch", "Cooked rice", 150),
    ("Lunch", "Dal, cooked", 150),
    ("Dinner", "Paneer", 100),
]

rows = []
for meal_type, name, qty in example_entries:
    food_row = foods.loc[foods["food_name"] == name].iloc[0]
    values = calculate_nutrients(food_row, qty)
    rows.append({"meal_type": meal_type, "food_name": name, "quantity_g": qty, **values})

example_meals = pd.DataFrame(rows)
example_meals


In [ ]:
daily_totals = example_meals[["calories_kcal", "protein_g", "carbs_g", "fat_g"]].sum().round(2)
daily_totals


## 8. Weekly Analytics Demonstration
The app summarizes saved entries by date. This example illustrates aggregation from sample records.

In [ ]:
# Demonstration data only; in the app, these values are derived from persisted meal logs.
weekly_demo = pd.DataFrame({
    "day": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    "protein_g": [72, 88, 94, 81, 105, 91, 78],
    "calories_kcal": [1800, 2050, 2140, 1930, 2250, 2100, 1880],
})
weekly_demo


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weekly_demo["day"], weekly_demo["protein_g"], marker="o")
ax.axhline(100, linestyle="--", label="Illustrative target: 100 g")
ax.set_title("Example weekly protein intake")
ax.set_ylabel("Protein (g)")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Running the Interactive Application
Install dependencies from `requirements.txt`, then run the following command from the project directory:

`streamlit run app.py`

The application provides tabs for the dashboard, meal logging, food comparison, target management/food database, and meal history/weekly analytics.

## 10. Testing Plan
- Verify that quantity must be positive.
- Check nutrient calculation for a known quantity (e.g., 200 g must be twice the per-100-g value).
- Confirm meal entries remain available after restarting the app.
- Confirm target changes are saved.
- Compare foods using the same per-100-g basis.
- Test empty-history and invalid-food-input cases.

## 11. Limitations and Ethical Considerations
- The included dataset is small and uses approximate illustrative values; it should be replaced or supplemented with verified values and source metadata before real-world use.
- Nutritional values vary by brand, preparation, edible portion, and moisture content.
- Targets are user-defined or estimates, not medical advice.
- The prototype does not diagnose conditions or prescribe diets.
- Days with no meal entries must not be interpreted as days with zero food intake.

## 12. Conclusion
FitNutrition demonstrates how Python can be used to combine data processing, nutritional calculations, persistent storage, visual analytics, and an interactive application. The project provides a foundation that can be extended with a larger verified food dataset, export features, and optional food–nutrient graph analysis.
